# Preparación de datos -  AIRBNB

1. Integración de los datos
2. Eliminar variables irrelevantes y redundantes
3. Descripción estadística de los datos
4. Limpieza de datos atípicos-errores
5. Limpieza de datos nulos
6. Análisis de correlaciones para redundancias entre todas las variables
7. Análisis de correlaciones para irrelevancias respecto a la objetivo
8. Guardar los datos preparados

*Nota: el balanceo y las transformaciones (ingeniería de características) no se incluyen en este notebook; se realizan en la etapa siguiente.*

In [ ]:
#Importamos librerías básicas
import pandas as pd # manipulacion dataframes
import numpy as np  # matrices y vectores
import matplotlib.pyplot as plt #gráfica

# 1. Integración de Datos

In [ ]:
data = pd.read_csv("AB_NYC_2019.csv")
data.head()

In [ ]:
data.info()

In [ ]:
#Variables categoricas
for col in data.columns:
    if data[col].dtype == 'object':
        data[col] = data[col].astype('category')

data.info()

In [ ]:
#Revisamos si tenemos ID repetidos
ids = data['id'].value_counts()
ids[ids > 1]

#Sólo dejamos el id mas reciente
data.drop_duplicates(subset=['id'], keep='last', inplace=True)

#Verificamos de nuevo si tenemos ID repetidos
ids = data['id'].value_counts()
ids[ids > 1]

# 2. Eliminar variables irrelevantes y redundantes
* Irrelevantes: cedulas, ids, nombres, telefonos, direcciones, códigos.
* Redundantes: validación por fórmula matemática que están repetidas

In [ ]:
# Irrelevantes: identificadores y nombres (id, name, host_id, host_name)
# Redundante: neighbourhood (queda determinado por latitude y longitude)
data = data.drop(['id', 'name', 'host_id', 'host_name', 'neighbourhood'], axis=1)  # axis=1 indica que son columnas
data.head()

Se elimina neighbourhood por redundancia: el barrio queda determinado por las coordenadas (latitud y longitud) y su agrupación general se conserva en neighbourhood_group. Además, su alta cardinalidad (más de 200 categorías) dificultaría su codificación.

# 3. Descripción estadística

**Variables numéricas**

In [ ]:
data.describe()

In [ ]:
#Histogramas
data.hist()

In [ ]:
data.plot.box()

In [ ]:
data['price'].plot.box()


In [ ]:
data['minimum_nights'].plot.box()

**Variables categóricas** (value_counts())

In [ ]:
data['neighbourhood_group'].value_counts().plot(kind='bar')

In [ ]:
data['room_type'].value_counts().plot(kind='barh')

In [ ]:
# Convertimos la columna a tipo datetime para poder extraer componentes temporales
data['last_review'] = pd.to_datetime(data['last_review'])

# Extraemos el año de la última reseña y graficamos su frecuencia de manera legible
plt.figure(figsize=(10, 5))
data['last_review'].dt.year.value_counts().sort_index().plot(kind='bar', color='skyblue')
plt.title('Distribución de Últimas Reseñas por Año')
plt.xlabel('Año de la última reseña')
plt.ylabel('Cantidad de publicaciones')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

**Gráficas de las relaciones entre variables**

In [ ]:
# Gráficas para identificar relaciones entre las variables
pd.plotting.scatter_matrix(data, figsize=(10,10))

**Perfilado de datos**

In [ ]:
!pip install ydata-profiling
from ydata_profiling import ProfileReport
profile_data=ProfileReport(data, minimal=True) # minimal=True
profile_data

In [ ]:
#Guardamos en html el perfilado de datos
profile_data.to_file(output_file="output.html")

# 4. Limpieza de datos atípicos-errores
- Sólo se limpian errores

Se ha decidido conservar los valores atípicos (outliers) detectados en la variable objetivo price y en otras características estructurales del conjunto de datos, en lugar de eliminarlos o imputarlos. Esto se debe a que dichos extremos tales como propiedades con tarifas por noche excepcionalmente elevadas— corresponden a fenómenos de mercado reales y legítimos (por ejemplo, alojamientos de lujo o villas exclusivas) y no a errores de captura o ruido instrumental. Eliminar o alterar estos registros mediante imputaciones artificiales restaría valor predictivo al modelo, distorsionando la variabilidad natural del mercado de alquileres en Nueva York y privando al algoritmo de información clave sobre el segmento de alta gama.

Sí se eliminan los registros que corresponden a **errores**:
* `price = 0`: un alojamiento no puede costar 0 dólares por noche.
* `minimum_nights > 365`: un mínimo de estadía superior a un año (con valores de hasta 1250 noches) no es coherente con un alquiler por noches y se considera un error de captura.

In [ ]:
# Guardamos la cantidad inicial de registros para comparar
registros_iniciales = len(data)

# Aplicamos las reglas de calidad de datos (sólo errores)
data = data[data['price'] > 0]
data = data[(data['minimum_nights'] > 0) & (data['minimum_nights'] <= 365)]

# Mostramos los resultados del filtrado
registros_finales = len(data)
print(f"Registros iniciales: {registros_iniciales}")
print(f"Registros después de la limpieza: {registros_finales}")
print(f"Registros eliminados: {registros_iniciales - registros_finales}")

data.head()

**5. Limpieza de datos nulos: Imputación**

Estrategia:
* Eliminar registros con mas de 15% de nulos
* Eliminar columnas con mas de 15%-20% de nulos
* Imputar por media, moda, mediana, vecinos cercanos. No se puede imputar más allá del 15% de los datos.
* Para casos especiales se crea modelo predictivo

In [ ]:
# Calculamos la cantidad y el porcentaje de valores nulos por columna
valores_nulos = data.isnull().sum()
porcentaje_nulos = (valores_nulos / len(data)) * 100

# Creamos un DataFrame para mostrar los resultados de manera ordenada
resumen_nulos = pd.DataFrame({
    'Valores Nulos': valores_nulos,
    'Porcentaje (%)': porcentaje_nulos
})

# Ordenamos de mayor a menor según el porcentaje de nulos
resumen_nulos = resumen_nulos.sort_values(by='Porcentaje (%)', ascending=False)

display(resumen_nulos)

**Tratamiento de los nulos**

* `reviews_per_month` y `last_review` tienen el mismo número de nulos (20.6 %). Se comprueba que son nulos **estructurales**: aparecen exactamente en los alojamientos con `number_of_reviews = 0`, es decir, que nunca han recibido reseñas.
* `reviews_per_month`: si no hay reseñas, el número de reseñas por mes es 0. Por lo tanto, no se trata de un dato perdido y se imputa con 0 (no aplica el límite del 15 %, porque el valor se conoce con certeza).
* `last_review`: es una fecha y no existe un valor con sentido para un alojamiento sin reseñas, por lo que la columna se elimina (supera el 15 %-20 % de nulos).

In [ ]:
# Verificamos que los nulos de reviews_per_month coinciden con los alojamientos sin reseñas
print("¿Nulos de reviews_per_month = alojamientos con 0 reseñas?:",
      (data['reviews_per_month'].isna() == (data['number_of_reviews'] == 0)).all())

# Imputamos con 0 (sin reseñas -> 0 reseñas por mes) y eliminamos last_review
data['reviews_per_month'] = data['reviews_per_month'].fillna(0)
data = data.drop(columns=['last_review'])

# Verificamos que ya no existan valores nulos en el dataset
print("\nValores nulos restantes por columna:")
print(data.isnull().sum())

# Visualizamos las dimensiones finales del dataset
print(f"\nDimensiones del dataset: {data.shape}")

In [ ]:
data.info()

# 6. Correlaciones para redundancias

In [ ]:
# Todas las variables deben ser numéricas para calcular las correlaciones
# Se crean dummies para las variables categóricas

data_num = pd.get_dummies(data, drop_first=False, dtype=int)
data_num.head()

In [ ]:
#Correlaciones
correlaciones=data_num.corr()
correlaciones

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Seleccionamos las columnas numéricas e incluimos las de baja cardinalidad convertidas a dummies
columnas_interes = ['latitude', 'longitude', 'price', 'minimum_nights', 'number_of_reviews', 'reviews_per_month', 'calculated_host_listings_count', 'availability_365']
data_reducida = pd.get_dummies(data[columnas_interes + ['neighbourhood_group', 'room_type']], drop_first=True, dtype=int)

# Calculamos la nueva matriz de correlación
corr_reducida = data_reducida.corr()

# Graficamos un mapa de calor legible y limpio
plt.figure(figsize=(12, 10))
sns.heatmap(
    corr_reducida,
    annot=True,
    fmt=".2f",
    cmap='coolwarm',
    linewidths=0.5,
    cbar_kws={"shrink": .8}
)

plt.title('Matriz de Correlación Simplificada y Legible', fontsize=16)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# 7. Correlaciones para irrelevancia con la variable objetivo

Se buscan correlaciones con la variable objetivo muy bajas (0.0-0.05 ) ->ultima fila

In [ ]:
#Correlaciones con la variable de objetivo
cor_variable_obj=correlaciones.loc['price']
cor_variable_obj

Para variables con 3 o más categorias se calcula promedio ponderado de las dummies que corresponden a la misma variable

In [ ]:
# 1. Identificamos variables numéricas directas
vars_numericas = ['latitude', 'longitude', 'minimum_nights', 'number_of_reviews', 'reviews_per_month', 'calculated_host_listings_count', 'availability_365']
corr_numericas = correlaciones.loc[vars_numericas, 'price'].abs().to_dict()

# 2. Función para calcular el promedio ponderado de las correlaciones absolutas de las dummies de una variable
def promedio_ponderado_corr(prefijo_variable):
    # Filtrar las columnas dummy de esta variable
    columnas_dummy = [col for col in data_num.columns if col.startswith(prefijo_variable + '_')]

    # Frecuencias (pesos) de cada categoría
    pesos = data_num[columnas_dummy].sum()
    total_pesos = pesos.sum()

    if total_pesos == 0:
        return 0

    # Correlaciones absolutas de estas dummies con price
    corrs = correlaciones.loc[columnas_dummy, 'price'].abs()

    # Promedio ponderado
    promedio_ponderado = np.average(corrs, weights=pesos)
    return promedio_ponderado

# 3. Calculamos la correlación consolidada para las variables categóricas
corr_categoricas = {
    'neighbourhood_group': promedio_ponderado_corr('neighbourhood_group'),
    'room_type': promedio_ponderado_corr('room_type')
}

# 4. Unimos ambos análisis de correlación absoluta con la variable objetivo
corr_consolidada = pd.Series({**corr_numericas, **corr_categoricas}).sort_values(ascending=True)

# 5. Graficamos el resultado de manera clara e impecable
plt.figure(figsize=(10, 6))
barras = plt.barh(corr_consolidada.index, corr_consolidada.values, color='skyblue', edgecolor='black')

# Añadimos los valores numéricos sobre cada barra para facilitar la lectura
for barra in barras:
    ancho = barra.get_width()
    plt.text(ancho + 0.002, barra.get_y() + barra.get_height()/2, f'{ancho:.4f}',
             va='center', ha='left', fontsize=10, fontweight='bold')

plt.title('Correlación Absoluta Consolidada con la Variable Objetivo (price)', fontsize=14, pad=15)
plt.xlabel('Correlación Absoluta (Promedio Ponderado para Categóricas)', fontsize=12)
plt.xlim(0, max(corr_consolidada.values) * 1.15)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

**Verificación con correlación de Spearman**

La correlación de Pearson sólo mide relaciones lineales y es muy sensible a los valores extremos; el precio presenta una distribución muy sesgada (valores de hasta 10 000). Por ello, antes de descartar una variable por irrelevancia, se verifica también la correlación de Spearman, que se basa en rangos y detecta relaciones monótonas no lineales.

In [ ]:
vars_revisar = ['latitude', 'longitude', 'minimum_nights', 'number_of_reviews',
                'reviews_per_month', 'calculated_host_listings_count', 'availability_365']

comparacion = pd.DataFrame({
    'Pearson': data[vars_revisar].corrwith(data['price']),
    'Spearman': data[vars_revisar].corrwith(data['price'], method='spearman')
}).round(3)

comparacion['|Pearson| < 0.05'] = comparacion['Pearson'].abs() < 0.05
comparacion['|Spearman| < 0.05'] = comparacion['Spearman'].abs() < 0.05
comparacion

**Conclusión del análisis de irrelevancia**

* **`latitude`**: presenta una correlación lineal baja con el precio (Pearson ≈ 0.03). Sin embargo, no se elimina por irrelevancia: su correlación de Spearman es mayor (≈ 0.14) y, sobre todo, su efecto sobre el precio no es lineal y depende de su combinación con la longitud (por ejemplo, la cercanía al centro de Manhattan). Ambas coordenadas se conservan y se transformarán en una variable de ubicación en la etapa de ingeniería de características.
* **`minimum_nights`, `number_of_reviews` y `reviews_per_month`**: sus correlaciones de Pearson están en el límite de 0.05, pero con Spearman todas superan ese umbral (≈ 0.10, -0.05 y -0.06). Por lo tanto, se conservan.
* **`neighbourhood_group` y `room_type`**: su correlación consolidada (promedio ponderado de las dummies) supera 0.05; se conservan.

En consecuencia, **no se elimina ninguna variable por irrelevancia**. Las variables categóricas se guardan sin codificar: la codificación (dummies) se realizó sólo para calcular las correlaciones y se aplicará en la etapa de transformaciones.

In [ ]:
# Mostramos las dimensiones y columnas finales del dataset preparado
print(f"Dimensiones actuales del dataset: {data.shape}")
print("\nColumnas finales en el conjunto de datos:")
print(list(data.columns))
display(data.head())

# 8. Guardar los datos preparados

In [ ]:
# Guardamos el conjunto de datos preparado en un archivo CSV en el almacenamiento local de Colab
data.to_csv('Airbnb_NYC_2019_limpio.csv', index=False)

print("¡Datos preparados guardados con éxito como 'Airbnb_NYC_2019_limpio.csv'!")
print("Puedes descargarlo haciendo clic en el panel izquierdo de archivos, buscando el archivo y seleccionando 'Descargar'.")